In [7]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cross_decomposition import CCA
from scipy.stats import pearsonr 
from scipy import stats
from typing import List, Tuple, Optional, Dict

###############################################################################
# Paths & constants
###############################################################################
PUPIL_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara")
EEG_ROOT   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_processed2")

FRONTAL_MIDLINE = ['AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4']

#OUT_TRIALS  = Path("trial_level_cca_fast.csv")
#OUT_SUBJECT = Path("subject_level_cca_fast.csv")
#OUT_WEIGHTS = Path("cca_weights_slow")

SUBJECTS = np.setdiff1d(np.arange(32, 99), [32, 37, 53, 61, 66, 78, 84, 90, 94, 96])
WIN_OFFSET1 = 200                  # discard first 200 ms
WIN_OFFSET2 = 110                  # discard last 110 ms
win = slice(WIN_OFFSET1, -WIN_OFFSET2)

###############################################################################
# Helper functions
###############################################################################

def normalise_eeg(x: np.ndarray) -> np.ndarray:
    """Centre each channel and scale so Σ x² = 1 over timexchannels."""
    x = x - x.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.mean(x**2))
    return x / scale


def cca_corr(eeg: np.ndarray, pupil: np.ndarray) -> float:
    """Canonical correlation (single component)."""
    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(eeg, pupil)
    u, v = cca.transform(eeg, pupil)
    return float(np.corrcoef(u[:, 0], v[:, 0])[0, 1])

def candidate_lags_units(step_ms=100, max_ms=1000):
    # Your code uses 10 ms units (lag_ms = shift*10)
    step_units = step_ms // 10
    max_units  = max_ms  // 10
    return list(range(-max_units, max_units + 1, step_units))
        
def load_all_trials(
        sub: int,
        eeg_root: Path = EEG_ROOT,
        pupil_root: Path = PUPIL_ROOT,
        min_len: int = 40,
        max_len_diff: int = 30,
) -> list[tuple[np.ndarray, np.ndarray, dict]]:
    """
    Load *all* valid EEG-pupil trial pairs for one subject.

    Parameters
    ----------
    sub : int
        Numeric subject ID (e.g. 42).
    eeg_root, pupil_root : Path
        Roots of the pre-processed EEG and pupil folders.
    min_len : int
        Minimum number of samples a pupil trace must have to be accepted.
    max_len_diff : int
        Reject trial if |len(pupil)-len(eeg)| exceeds this.
    Returns
    -------
    trials : list of (eeg, pupil_z, meta)
        * eeg        - (T x n_channels) float64, already centred/scaled
        * pupil_z    - (T x 1) float64, per-trial z-scored
        * meta       - dict with subject/condition/load/epoch
    """
    trials = []
    sub_tag = f"sub-{sub:03d}"
    eeg_sub  = eeg_root   / sub_tag
    pupil_sub = pupil_root / sub_tag

    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing - skipped")
        return trials

    # iterate condition (“control” / “memory”) and load (“05” / “09” / “13”)
    for cond_path in sorted(eeg_sub.iterdir()):
        if not cond_path.is_dir():
            continue
        for load_path in sorted(cond_path.iterdir()):
            if not load_path.is_dir():
                continue

            # matching pupil directory
            pupil_path = pupil_sub / cond_path.name / load_path.name
            if not pupil_path.exists():
                continue

            eeg_epochs   = sorted(load_path.glob("trial_*.csv"))
            pupil_epochs = sorted(pupil_path.glob("trial_*.csv"))
            common = {f.name for f in eeg_epochs} & {f.name for f in pupil_epochs}
            if not common:
                continue

            for fname in sorted(common):
                eeg_df = pd.read_csv(load_path / fname, comment="#", index_col=0)
                pupil_df = pd.read_csv(pupil_path / fname, comment="#", skiprows=1,
                                       names=["time", "diameter_z"], index_col=0)

                eeg   = eeg_df.values.astype(float)
                pupil = pupil_df["diameter_z"].values.astype(float)

                # basic validity checks
                if len(pupil) < min_len or abs(len(pupil) - len(eeg)) > max_len_diff:
                    continue

                # normalise signals ----------------------------------------
                eeg_norm = normalise_eeg(eeg)           # your helper from before
                pupil_z  = ((pupil - pupil.mean()) / pupil.std(ddof=0))

                # same number of samples
                T = min(len(eeg_norm), len(pupil_z))
                eeg_norm = eeg_norm[0:T, :]  # (T x n_channels)
                pupil_z  = pupil_z[0:T].reshape(-1, 1)

                meta = {
                    "subject":   sub_tag,
                    "condition": cond_path.name,
                    "load":      int(load_path.name),
                    "epoch":     fname
                }
                trials.append((eeg_norm, pupil_z, meta))

    return trials


In [8]:
# ================================
# CCA on panel MEANS (no concat)
# ================================
from sklearn.cross_decomposition import CCA
from scipy.stats import pearsonr
from collections import defaultdict
import matplotlib.pyplot as plt

OUT_PANEL = Path("cca_both_means2.csv")   # conditionxload results
OUT_WEIGHTS_DIR = Path("cca_weights_both2_means")
OUT_WEIGHTS_DIR.mkdir(exist_ok=True, parents=True)

#OUT_PANEL_SUBJ = Path("cca_both_means_per_subject2.csv")
#OUT_WEIGHTS_DIR_SUBJ = Path("cca_weights_both_mean_subj2")
#OUT_WEIGHTS_DIR_SUBJ.mkdir(exist_ok=True, parents=True)

PLOT_DIR = Path("plot_cca_lag_means_both2")

def _panel_mean_pair(trials, shift: int = 0, win: slice | None = None) -> tuple[np.ndarray, np.ndarray]:
    """
    Build the panel MEAN pair (X̄, ȳ) from a list of trials:
      X̄: (L x C) = mean over trials of rolled/cropped EEG
      ȳ: (L x 1) = mean over trials of cropped pupil
    All trials are first aligned to the shortest length after rolling + windowing.
    """
    eeg_list, pup_list = [], []
    for eeg, pupil, _ in trials:
        eeg_shift = np.roll(eeg, shift, axis=0) if shift else eeg
        pup1d = pupil[:, 0] if pupil.ndim == 2 else pupil

        T = min(len(eeg_shift), len(pup1d))
        eeg_shift = eeg_shift[:T]
        pup1d     = pup1d[:T]

        if win is not None:
            w = win if isinstance(win, slice) else slice(win[0], win[1])
            eeg_shift = eeg_shift[w]
            pup1d     = pup1d[w]

        eeg_list.append(eeg_shift)
        pup_list.append(pup1d)

    if not eeg_list:
        return None, None

    # align to shortest length across trials
    Lmin = min(len(x) for x in eeg_list)
    C    = eeg_list[0].shape[1]
    X = np.stack([x[:Lmin] for x in eeg_list], axis=0)        # (N, L, C)
    y = np.stack([p[:Lmin] for p in pup_list], axis=0)        # (N, L)

    X_bar = X.mean(axis=0)                                    # (L, C)
    y_bar = y.mean(axis=0).reshape(-1, 1)                     # (L, 1)
    return X_bar, y_bar

def search_best_lag_on_means(
        trials, shifts: list[int], win: slice | None = None, return_curve: bool = False
) -> tuple[float, int, dict[int, float] | None]:
    """Grid search lag that maximizes CCA r on the PANEL MEANS."""
    r_per_shift = {}
    for s in shifts:
        X_bar, y_bar = _panel_mean_pair(trials, shift=s, win=win)
        if X_bar is None:
            r_per_shift[s] = np.nan
            continue
        r_per_shift[s] = cca_corr(X_bar, y_bar)
    # pick best valid
    valid = {k: v for k, v in r_per_shift.items() if np.isfinite(v)}
    best_shift = max(valid, key=valid.get)
    best_r = valid[best_shift]
    return (best_r, best_shift, r_per_shift if return_curve else None)

def save_panel_weights(cca: CCA, channel_names: list[str], tag: str):
    """Save EEG and pupil weights for the panel (conditionxload)."""
    wX = cca.x_weights_.ravel()   # (C,)
    wY_og = cca.y_weights_.ravel()   # (1,)
    wX = wX / wY_og[0]   # normalize so pupil weight = 1
    wY = 1.0
    df = pd.DataFrame({"channel": channel_names, "weight": wX})
    df.to_csv(OUT_WEIGHTS_DIR / f"eeg_weights_{tag}.csv", index=False)
    pd.DataFrame({"pupil_weight":[wY]}).to_csv(OUT_WEIGHTS_DIR / f"pupil_weight_{tag}.csv", index=False)

def _plot_lag_curve(r_per_shift: dict[int, float], best_shift: int, subject: int, cond: str, load: int, out_dir: Path):
    """Save an r-vs-lag plot highlighting the best shift."""
    lags = np.array(sorted(r_per_shift.keys()))
    rs   = np.array([r_per_shift[s] for s in lags])
    lag_ms = lags * 10

    fig, ax = plt.subplots(figsize=(5.0, 3.2))
    ax.plot(lag_ms, rs, lw=2)
    ax.axvline(best_shift * 10, ls="--", lw=1.2, color="k", label=f"best = {best_shift*10:+} ms")
    # annotate best point
    best_idx = np.where(lags == best_shift)[0][0]
    ax.plot([lag_ms[best_idx]], [rs[best_idx]], "o")
    ax.set_xlabel("Lag (ms)")
    ax.set_ylabel("CCA r")
    if subject > 0:
        ax.set_title(f"sub-{subject:03d} | {cond} L{load:02d}")
    else:
        ax.set_title(f"ALL subjects | {cond} L{load:02d}")
    ax.grid(alpha=.3)
    ax.legend(frameon=False)
    fig.tight_layout()
    if subject < 0:
        fname = out_dir / f"lagcurve_sub-{subject:03d}_{cond}_L{load:02d}.png"
    else:
        fname = out_dir / f"lagcurve_sub-All_{cond}_L{load:02d}.png"
    fig.savefig(fname, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return fname

# ---------------------------
# Run across condition x load
# ---------------------------
def run_panel_mean_cca(
        subjects=SUBJECTS,
        loads=(5, 9, 13),
        shifts_units=None,          # 10 ms units at 100 Hz
        win: slice = slice(WIN_OFFSET1, -WIN_OFFSET2),
        save_weights=True,
        make_plots=True,
):
    if shifts_units is None:
        shifts_units = candidate_lags_units(step_ms=100, max_ms=2500)  # e.g., every 100 ms within ±1000 ms


    # collect ALL trials across selected subjects
    all_trials = []
    for s in subjects:
        print(f"Loading subject {s} ...")
        all_trials.extend(load_all_trials(int(s)))

    # bucket by (condition, load)
    buckets = defaultdict(list)
    for eeg, pupil, meta in all_trials:
        cond = meta["condition"].lower()
        load = int(meta["load"])
        if load in loads and cond in ("memory", "control"):
            buckets[(cond, load)].append((eeg, pupil, meta))

    rows = []
    for cond in ("memory", "control"):
        for load in loads:
            trials = buckets.get((cond, load), [])
            if not trials:
                print(f"No trials for {cond} load {load}")
                continue

            # 1) best lag on MEANS
            best_r, best_shift_u, r_curve = search_best_lag_on_means(trials, shifts_units, win=win, return_curve=True)
            lag_ms = best_shift_u * 10

            # 2) fit CCA on the MEANS at best lag and report r (again)
            X_bar, y_bar = _panel_mean_pair(trials, shift=best_shift_u, win=win)
            cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
            cca.fit(X_bar, y_bar)
            u, v = cca.transform(X_bar, y_bar)
            r_final, p_final = pearsonr(u[:, 0], v[:, 0])

            # 3) save weights (optional)
            if save_weights:
                tag = f"{cond}_L{load}_lag{lag_ms}ms"
                save_panel_weights(cca, FRONTAL_MIDLINE, tag)

                # 4) Plot r vs lag (optional)
                if make_plots and r_curve:
                    plot_dir = PLOT_DIR
                    plot_dir.mkdir(exist_ok=True, parents=True)
                    plot_path = _plot_lag_curve(r_curve, best_shift_u, int(0), cond, int(load), plot_dir)

            # 4) record panel result
            n_trials = len(trials)
            n_subjects = len({t[2]["subject"] for t in trials})
            rows.append({
                "condition": cond,
                "load": load,
                "lag_ms": lag_ms,
                "r": float(r_final),
                "p_value": float(p_final),
                "n_trials": n_trials,
                "n_subjects": n_subjects,
            })
            print(f"{cond:7s} L{load:02d}: best lag {lag_ms:+4d} ms, r={r_final:.3f} (Ntr={n_trials}, Nsub={n_subjects})")

    df = pd.DataFrame(rows).sort_values(["condition", "load"])
    df.to_csv(OUT_PANEL, index=False)
    print(f"Saved panel results → {OUT_PANEL}")
    return df

# ---- run it ----
# df_panel = run_panel_mean_cca()


In [9]:
df_panel = run_panel_mean_cca()

Loading subject 33 ...
Loading subject 34 ...
Loading subject 35 ...
Loading subject 36 ...
Loading subject 38 ...
Loading subject 39 ...
Loading subject 40 ...
Loading subject 41 ...
Loading subject 42 ...
Loading subject 43 ...
Loading subject 44 ...
Loading subject 45 ...
Loading subject 46 ...
Loading subject 47 ...
Loading subject 48 ...
Loading subject 49 ...
Loading subject 50 ...
Loading subject 51 ...
Loading subject 52 ...
Loading subject 54 ...
Loading subject 55 ...
Loading subject 56 ...
Loading subject 57 ...
Loading subject 58 ...
Loading subject 59 ...
Loading subject 60 ...
Loading subject 62 ...
Loading subject 63 ...
Loading subject 64 ...
Loading subject 65 ...
Loading subject 67 ...
Loading subject 68 ...
Loading subject 69 ...
Loading subject 70 ...
Loading subject 71 ...
Loading subject 72 ...
Loading subject 73 ...
Loading subject 74 ...
Loading subject 75 ...
Loading subject 76 ...
Loading subject 77 ...
Loading subject 79 ...
Loading subject 80 ...
Loading sub

In [ ]:
import matplotlib.pyplot as plt
# ----------------------- config / outputs -----------------------
PLOT_DIR = Path("plots_cca_lag_curves_subject_both")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def save_panel_weights_subject(cca: CCA, channel_names: list[str], subject: int, cond: str, load: int, lag_ms: int):
    tag = f"sub-{subject:03d}_{cond}_L{load}_lag{lag_ms}ms"
    wX = cca.x_weights_.ravel()   # (C,)
    wY_og = cca.y_weights_.ravel()   # (1,)
    wX = wX / wY_og[0]   # normalize so pupil weight = 1
    wY = 1.0
    pd.DataFrame({"channel": channel_names, "weight": wX}).to_csv(OUT_WEIGHTS_DIR_SUBJ / f"eeg_weights_{tag}.csv", index=False)
    pd.DataFrame({"pupil_weight": [wY]}).to_csv(OUT_WEIGHTS_DIR_SUBJ / f"pupil_weight_{tag}.csv", index=False)


# ----------------------- main runner ----------------------------
def run_panel_mean_cca_per_subject(
        subjects=None,
        loads=(5, 9, 13),
        shifts_units=None,             # 10 ms units at 100 Hz
        win: slice = slice(WIN_OFFSET1, -WIN_OFFSET2),
        save_weights=True,
        weights_dir: Path = Path("cca_weights_panel_subject"),
        make_plots=True,
        plot_dir: Path = PLOT_DIR
):
    """
    Per SUBJECT × (condition, load):
      • pool trials → mean EEG (L×C), mean pupil (L×1)
      • grid-search best lag on the means
      • fit CCA at best lag, record stats
      • optionally: save weights + an r-vs-lag plot marking best lag
    """
    if shifts_units is None:
        shifts_units = candidate_lags_units(step_ms=100, max_ms=2000)
    if subjects is None:
        subjects = SUBJECTS

    if make_plots:
        plot_dir.mkdir(parents=True, exist_ok=True)
    if save_weights:
        weights_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for sub in subjects:
        trials = load_all_trials(int(sub))
        if not trials:
            print(f"sub-{sub:03d}: no trials")
            continue

        buckets = defaultdict(list)
        for eeg, pupil, meta in trials:
            cond = meta["condition"].lower()
            load = int(meta["load"])
            if cond in ("memory", "control") and load in loads:
                buckets[(cond, load)].append((eeg, pupil, meta))

        for cond in ("memory", "control"):
            for load in loads:
                sub_trials = buckets.get((cond, load), [])
                if not sub_trials:
                    continue

                # 1) best lag + full curve
                best_r, best_shift_u, r_curve = search_best_lag_on_means(sub_trials, shifts_units, win=win, return_curve=True)
                lag_ms = int(best_shift_u * 10)

                # 2) Fit at best lag and compute final stats
                X_bar, y_bar = _panel_mean_pair(sub_trials, shift=best_shift_u, win=win)
                cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
                cca.fit(X_bar, y_bar)
                u, v = cca.transform(X_bar, y_bar)
                r_final, p_final = pearsonr(u[:, 0], v[:, 0])

                # 3) Save weights (optional)
                if save_weights:
                    save_panel_weights_subject(cca, FRONTAL_MIDLINE, int(sub), cond, int(load), lag_ms)

                # 4) Plot r vs lag (optional)
                plot_path = None
                if make_plots and r_curve:
                    plot_path = _plot_lag_curve(r_curve, best_shift_u, int(sub), cond, int(load), plot_dir)

                rows.append({
                    "subject": int(sub),
                    "condition": cond,
                    "load": int(load),
                    "lag_ms": lag_ms,
                    "r": float(r_final),
                    "p_value": float(p_final),
                    "n_trials": len(sub_trials),
                    "curve_png": str(plot_path) if plot_path else ""
                })
                print(f"sub-{sub:03d}  {cond:7s} L{load:02d}  lag={lag_ms:+4d} ms  r={r_final:.3f}  (Ntr={len(sub_trials)})")

    df = pd.DataFrame(rows).sort_values(["subject", "condition", "load"])
    df.to_csv(OUT_PANEL_SUBJ, index=False)
    print(f"Saved per-subject panel results → {OUT_PANEL_SUBJ}")
    print(f"Lag curves saved to → {PLOT_DIR.resolve()}")
    return df

# ---- run it ----
# df_subj_panels = run_panel_mean_cca_per_subject()

In [59]:
df_subj_panels = run_panel_mean_cca_per_subject()

sub-033  memory  L05  lag=-1500 ms  r=0.804  (Ntr=35)
sub-033  memory  L09  lag=-400 ms  r=0.601  (Ntr=35)
sub-033  memory  L13  lag=+2000 ms  r=0.507  (Ntr=36)
sub-033  control L05  lag=+1600 ms  r=0.674  (Ntr=18)
sub-033  control L09  lag=+1200 ms  r=0.563  (Ntr=18)
sub-033  control L13  lag=+1800 ms  r=0.578  (Ntr=18)
sub-034  memory  L05  lag=+200 ms  r=0.814  (Ntr=35)
sub-034  memory  L09  lag=+1800 ms  r=0.770  (Ntr=30)
sub-034  memory  L13  lag=+1900 ms  r=0.785  (Ntr=32)
sub-034  control L05  lag=-1900 ms  r=0.753  (Ntr=15)
sub-034  control L09  lag=+2000 ms  r=0.627  (Ntr=14)
sub-034  control L13  lag=+2000 ms  r=0.482  (Ntr=16)
sub-035  memory  L05  lag=+1100 ms  r=0.712  (Ntr=11)
sub-035  memory  L09  lag=+2000 ms  r=0.571  (Ntr=4)
sub-035  memory  L13  lag=-1100 ms  r=0.525  (Ntr=1)
sub-035  control L05  lag=  +0 ms  r=0.688  (Ntr=5)
sub-035  control L09  lag=-1800 ms  r=0.603  (Ntr=5)
sub-035  control L13  lag=-700 ms  r=0.541  (Ntr=5)
sub-036  memory  L05  lag=+1700 ms  r

In [54]:
import pandas as pd
from pathlib import Path
import numpy as np

# ----- input files -----
p_fast = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\cca_fast_means_per_subject.csv")
p_slow = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\cca_slow_means_per_subject.csv")
p_both = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\cca_both_means_per_subject.csv")

# ----- output file -----
out_path = p_both.parent / "cca_fast_vs_slow_vs_both_by_subject.csv"

# Common column schema of your files:
COLS = ["subject", "condition", "load", "lag_ms", "r", "p_value", "n_trials"]

def load_panel_csv(path: Path) -> pd.DataFrame:
    """Read a panel CSV that may or may not have a header; coerce types safely."""
    df = pd.read_csv(path, header=None, names=COLS)
    # If the file actually had a header, the first row will be strings. Drop it.
    # Keep only rows where subject is numeric.
    df = df[pd.to_numeric(df["subject"], errors="coerce").notna()].copy()
    # Coerce dtypes
    df["subject"]   = df["subject"].astype(int)
    df["condition"] = df["condition"].astype(str).str.lower().str.strip()
    df["load"]      = pd.to_numeric(df["load"], errors="coerce").astype(int)
    df["lag_ms"]    = pd.to_numeric(df["lag_ms"], errors="coerce").astype(int)
    # r, p_value, n_trials not strictly needed, but coerce anyway
    for c in ["r", "p_value"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["n_trials"] = pd.to_numeric(df["n_trials"], errors="coerce").astype("Int64")
    return df

# Load all three
df_fast = load_panel_csv(p_fast).rename(columns={"lag_ms": "lag_ms_fast"})
df_slow = load_panel_csv(p_slow).rename(columns={"lag_ms": "lag_ms_slow"})
df_both = load_panel_csv(p_both).rename(columns={"lag_ms": "lag_ms_both"})

# Keep only memory condition
df_fast = df_fast[df_fast["condition"] == "memory"]
df_slow = df_slow[df_slow["condition"] == "memory"]
df_both = df_both[df_both["condition"] == "memory"]

# Keep only the key columns we’ll merge on + their lags
keys = ["subject", "condition", "load"]
df_fast_k = df_fast[keys + ["lag_ms_fast"]]
df_slow_k = df_slow[keys + ["lag_ms_slow"]]
df_both_k = df_both[keys + ["lag_ms_both"]]

# Merge on subject × condition × load
merged = (df_both_k
          .merge(df_fast_k, on=keys, how="outer")
          .merge(df_slow_k, on=keys, how="outer"))

# Compute absolute differences to "both"
merged["diff_fast_ms"] = (merged["lag_ms_fast"] - merged["lag_ms_both"]).abs()
merged["diff_slow_ms"] = (merged["lag_ms_slow"] - merged["lag_ms_both"]).abs()

# Decide which is closer (handle ties)
def closer_row(row):
    a, b = row["diff_fast_ms"], row["diff_slow_ms"]
    if pd.isna(a) and pd.isna(b):  return "missing"
    if pd.isna(a):                 return "slow"
    if pd.isna(b):                 return "fast"
    if a < b:                      return "fast"
    if b < a:                      return "slow"
    return "tie"

merged["closer_to_both"] = merged.apply(closer_row, axis=1)

# Order columns nicely
merged = merged.sort_values(keys).reset_index(drop=True)
merged = merged[keys + ["lag_ms_slow", "lag_ms_fast", "lag_ms_both", "closer_to_both"]]

# Save
merged.to_csv(out_path, index=False)
print(f"Saved merged comparison → {out_path}")
print(merged.head(12))


Saved merged comparison → C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\cca_fast_vs_slow_vs_both_by_subject.csv
    subject condition  load  lag_ms_slow  lag_ms_fast  lag_ms_both  \
0        33    memory     5          900          300         -400   
1        33    memory     9         1000            0         -400   
2        33    memory    13        -1000        -1100          700   
3        34    memory     5         1000            0          200   
4        34    memory     9         1000        -1100          700   
5        34    memory    13         -700         -600          500   
6        35    memory     5          500            0         1000   
7        35    memory     9          500         -500          100   
8        35    memory    13         -600         -400        -1000   
9        36    memory     5          900          100          800   
10       36    memory     9         1000          600          600   
11  

In [17]:
from pathlib import Path
from PIL import Image
import re

# --- INPUT FOLDERS (edit if needed) ---
DIR_BOTH = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plot_cca_lag_means_both")
DIR_FAST = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plot_cca_lag_means_fast")
DIR_SLOW = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plot_cca_lag_means_slow")

# --- OUTPUT FOLDER ---
OUT_DIR  = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plots_all_means")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# loads to include:
LOADS = [5, 9, 13]

# pattern for memory files, e.g. "lagcurve_sub-033_memory_L05.png"
# pat = re.compile(r"lagcurve_sub-(\d{3})_memory_L(05|09|13)\.png", re.IGNORECASE)
pat = re.compile(r"lagcurve_sub-All_(memory|control)_L(05|09|13)\.png", re.IGNORECASE)

def candidates_from_folder(folder: Path):
    """Return set of (subject, load) pairs seen in this folder (memory only)."""
    pairs = set()
    for p in folder.glob("lagcurve_sub-All_*_L*.png"):
        m = pat.match(p.name)
        if m:
            cond = m.group(1)
            load = int(m.group(2))
            pairs.add((cond, load))
    return pairs

def stitch_triptych(img_paths, out_path: Path, pad=8, bg=(255, 255, 255)):
    """
    Place three images side-by-side, same height, with 'pad' pixels between.
    img_paths = [path_both, path_fast, path_slow]
    """
    imgs = [Image.open(p).convert("RGB") for p in img_paths]
    # unify height (keep aspect)
    target_h = min(im.size[1] for im in imgs)
    resized = []
    for im in imgs:
        w, h = im.size
        new_w = int(round(w * (target_h / h)))
        resized.append(im.resize((new_w, target_h), Image.LANCZOS))

    total_w = sum(im.size[0] for im in resized) + pad * 2   # two gaps between 3 images
    canvas = Image.new("RGB", (total_w, target_h), color=bg)

    x = 0
    for i, im in enumerate(resized):
        canvas.paste(im, (x, 0))
        x += im.size[0] + (pad if i < 2 else 0)

    canvas.save(out_path, quality=95)

# --- Build the set of (subject, load) to process (memory only) ---
pairs = set()
for folder in [DIR_BOTH, DIR_FAST, DIR_SLOW]:
    if folder.exists():
        pairs |= candidates_from_folder(folder)

# Optionally restrict to specific loads
pairs = {(s, L) for (s, L) in pairs if L in LOADS}

# --- Process each (subject, load) ---
missing = []
done = 0
for cond, load in sorted(pairs):
    fname = f"lagcurve_sub-All_{cond}_L{load:02d}.png"
    p_both = DIR_BOTH / fname
    p_fast = DIR_FAST / fname
    p_slow = DIR_SLOW / fname

    if not (p_both.exists() and p_fast.exists() and p_slow.exists()):
        missing.append((cond, load, [str(p) for p in [p_both, p_fast, p_slow] if not Path(p).exists()]))
        continue

    out = OUT_DIR / f"triptych_sub-{cond}_memory_L{load:02d}.png"
    stitch_triptych([p_both, p_fast, p_slow], out)
    done += 1
    print(f"Saved → {out}")

print(f"\nFinished. Created {done} triptych image(s) in: {OUT_DIR}")
if missing:
    print("Skipped due to missing source(s):")
    for cond, load, miss in missing:
        print(f"  condition-{cond}, L{load:02d}: missing -> {', '.join(miss)}")


Saved → C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plots_all_means\triptych_sub-control_memory_L05.png
Saved → C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plots_all_means\triptych_sub-control_memory_L09.png
Saved → C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plots_all_means\triptych_sub-control_memory_L13.png
Saved → C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plots_all_means\triptych_sub-memory_memory_L05.png
Saved → C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plots_all_means\triptych_sub-memory_memory_L09.png
Saved → C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\fast_slow_components\plots_all_means\triptych_sub-memory_memory_L13.png

Finished. Created 6 triptych image(s) in: C:\Users\cdd\Documents\Uni\Special_course\code\Special_cours